In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings


warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore")
pd.options.display.max_columns = None

In [ ]:
to_drop = ["Latitude", "Longitude", "datetime"]
target = "UHI Index"
solar_stat = "Max"

In [ ]:
osm_to_keep = pd.read_csv(
    "/content/drive/MyDrive/EY 2025/Train_OSM.csv", nrows=1
).columns
osm_to_keep = osm_to_keep[osm_to_keep.str.contains("natural")]
osm_to_keep

Index(['natural_tree_distance_mean', 'natural_tree_distance_max',
       'natural_tree_distance_min', 'natural_tree_distance_median',
       'natural_tree_count', 'natural_tree_distance_var',
       'natural_tree_distance_std', 'natural_stone_distance_mean',
       'natural_stone_distance_max', 'natural_stone_distance_min',
       'natural_stone_distance_median', 'natural_stone_count',
       'natural_stone_distance_var', 'natural_stone_distance_std',
       'natural_peak_distance_mean', 'natural_peak_distance_max',
       'natural_peak_distance_min', 'natural_peak_distance_median',
       'natural_peak_count', 'natural_peak_distance_var',
       'natural_peak_distance_std', 'natural_shrub_distance_mean',
       'natural_shrub_distance_max', 'natural_shrub_distance_min',
       'natural_shrub_distance_median', 'natural_shrub_count',
       'natural_shrub_distance_var', 'natural_shrub_distance_std'],
      dtype='object')

In [ ]:
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.ensemble import ExtraTreesRegressor


def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    insample = r2_score(y_train, model.predict(X_train))
    outsample = r2_score(y_test, model.predict(X_test))
    return insample, outsample


def load_data(typ="Train"):
    global to_drop

    df = (
        pd.concat(
            [
                pd.read_csv(f"/content/drive/MyDrive/EY 2025/{typ}_Final.csv"),
                pd.read_csv(
                    f"/content/drive/MyDrive/EY 2025/{typ}_Building_Features_Revamped.csv"
                ).fillna(0),
                pd.read_csv(
                    f"/content/drive/MyDrive/EY 2025/{typ}_SolarData{solar_stat}SinceTrainTime.csv"
                ),
                pd.read_csv(
                    f"/content/drive/MyDrive/EY 2025/{typ}_OSM.csv", usecols=osm_to_keep
                ),
            ],
            axis=1,
        )
        .drop(to_drop, axis=1, errors="ignore")
        .fillna(0)
    )
    df.drop(target, axis=1).columns
    df = df.pipe(add_features).replace(np.inf, np.nan).fillna(0)
    return df


def prepare_data(df, typ, train_size):
    X_train, X_test, y_train, y_test = create_train(df, train_size=train_size)
    return X_train, X_test, y_train, y_test


def add_features(df):
    stats = ["median", "mean", "min", "max", "var", "std"]
    epsilon = 1e-7

    count_cols = df.columns[df.columns.str.contains("nearby_building_count")]
    for col in count_cols:
        divider = int(col.split("_")[0].replace("m", "")[:-1])
        df[f"{col}_density_per_10m"] = df[col] / divider

    for stat in stats:
        df[f"{stat}_evi_x_lwir"] = df[f"evi_{stat}"] * df[f"lwir_{stat}"]
        df[f"{stat}_ndbi_x_lwir"] = df[f"ndbi_{stat}"] * df[f"lwir_{stat}"]
        df[f"{stat}_ndbi_/_ndwi"] = df[f"ndbi_{stat}"] / df[f"ndwi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_ndvi"] = df[f"ndbi_{stat}"] / df[f"ndvi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_evi"] = df[f"ndbi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_wvp_/_lwir"] = df[f"wvp_{stat}"] / df[f"lwir_{stat}"].add(epsilon)
        df[f"{stat}_infra_red_combo"] = (
            df[f"nir08_{stat}"] * df[f"swir16_{stat}"] * df[f"swir22_{stat}"]
        )
        df[f"{stat}_relative_ndvi"] = df[f"ndvi_{stat}"] / (
            df[f"ndvi_{stat}"].max() + epsilon
        )
        df[f"{stat}_relative_ndwi"] = df[f"ndwi_{stat}"] / (
            df[f"ndwi_{stat}"].max() + epsilon
        )
        df[f"{stat}_ndvi_ndwi_ratio"] = df[f"ndvi_{stat}"] / df[f"ndwi_{stat}"].add(
            epsilon
        )
        df[f"{stat}_ndvi_evi_ratio"] = df[f"ndvi_{stat}"] / df[f"evi_{stat}"].add(
            epsilon
        )
        df[f"{stat}_ndwi_evi_ratio"] = df[f"ndwi_{stat}"] / df[f"evi_{stat}"].add(
            epsilon
        )
        df[f"{stat}_ndvi_ndwi_diff"] = df[f"ndvi_{stat}"] - df[f"ndwi_{stat}"]
        df[f"{stat}_ndvi_evi_diff"] = df[f"ndvi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_ndwi_evi_diff"] = df[f"ndwi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_combined_spectral_index"] = (
            df[f"ndvi_{stat}"] + df[f"ndwi_{stat}"] + df[f"evi_{stat}"]
        ) / 3

        df[f"{stat}_bsi"] = (
            df[f"swir16_{stat}"] + df[f"swir22_{stat}"] - 2 * df[f"nir08_{stat}"]
        ) / (df[f"swir16_{stat}"] + df[f"swir22_{stat}"] + 2 * df[f"nir08_{stat}"])
        df[f"{stat}_savi"] = ((df[f"nir08_{stat}"] - df[f"red_{stat}"]) * (1 + 0.5)) / (
            df[f"nir08_{stat}"] + df[f"red_{stat}"] + 0.5
        )
        df[f"{stat}_sr"] = df[f"nir08_{stat}"] / df[f"red_{stat}"]
        df[f"{stat}_dsi"] = (df[f"swir22_{stat}"] - df[f"nir08_{stat}"]) / (
            df[f"swir22_{stat}"] + df[f"nir08_{stat}"]
        )
        df[f"{stat}_wvi"] = df[f"wvp_{stat}"] / (
            df[f"swir22_{stat}"] + df[f"swir16_{stat}"] + df[f"nir08_{stat}"]
        )

    df["mean_vci"] = (df["ndvi_mean"] - df["ndvi_min"]) / (
        df["ndvi_max"] - df["ndvi_min"]
    )
    df["median_vci"] = (df["ndvi_median"] - df["ndvi_min"]) / (
        df["ndvi_max"] - df["ndvi_min"]
    )

    return df


def create_train(df_features, target="UHI Index", train_size=0.8):
    X = df_features.drop(target, axis=1)
    y = df_features[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=train_size)

    return X_train, X_test, y_train, y_test


def round1(models, train_size, score_func=f_regression, percentile=30):
    separator = "-" * 66
    spaces = " " * 24
    equals = "=" * 32

    print(spaces, "Starting round 1", spaces)
    print(separator)
    df = load_data(typ="Train")
    X_train, X_test, y_train, y_test = prepare_data(
        df, typ="Train", train_size=train_size
    )
    select = SelectPercentile(score_func, percentile=percentile)
    select.fit(X_train, y_train)

    X_train = X_train[select.get_feature_names_out()]

    X_test = X_test[X_train.columns]

    for model in tqdm(models):
        insample, outsample = evaluate_model(
            model["model"], X_train, X_test, y_train, y_test
        )
        model["insample"] = insample
        model["outsample"] = outsample

    results = (
        pd.DataFrame(models)
        .sort_values("outsample", ascending=False)
        .reset_index(drop=True)
    )
    print(equals, "Model Scores", equals)
    print(results)
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    importance = (
        pd.DataFrame(
            {
                "Features": X_train.columns,
                "Importance": best_model.model.feature_importances_,
            },
        )
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )

    importance["cumulative_importance"] = (
        importance.Importance.cumsum() / importance.Importance.sum()
    ).round(2)
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return importance


def round2(models, pareto_threshold, importance, train_size):
    separator = "-" * 66
    spaces = " " * 24
    equals = "=" * 32

    if importance is not None:
        pareto = importance[importance.cumulative_importance <= pareto_threshold]
        print(equals, "Pareto Features", equals)
        print(pareto)

    print(separator)

    print(spaces, "Starting round 2", spaces)
    print(separator)

    df = load_data(typ="Train")
    use_cols = pareto.Features

    X_train, X_test, y_train, y_test = prepare_data(
        df, typ="Train", train_size=train_size
    )
    X_train = X_train[use_cols]
    X_test = X_test[X_train.columns]

    len(X_train.columns) // 3

    for model in tqdm(models):
        insample, outsample = evaluate_model(
            model["model"], X_train, X_test, y_train, y_test
        )
        model["insample"] = insample
        model["outsample"] = outsample

    results = (
        pd.DataFrame(models)
        .sort_values("outsample", ascending=False)
        .reset_index(drop=True)
    )
    print(equals, "Model Scores", equals)
    print(results.head(1))
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    importance = (
        pd.DataFrame(
            {
                "Features": X_train.columns,
                "Importance": best_model.model.feature_importances_,
            },
        )
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )
    importance["cumulative_importance"] = (
        importance.Importance.cumsum() / importance.Importance.sum()
    ).round(2)
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return best_model, X_train

In [ ]:
train_shape = load_data().shape
print(f"Train rows = {train_shape[0]:,}")

Train rows = 11,229


# Modelling

In [ ]:
basic_params = {
    "n_jobs": -1,
    "max_features": "log2",
}

models = [
    {"model": ExtraTreesRegressor(n_estimators=300, **basic_params)},
    {"model": ExtraTreesRegressor(n_estimators=275, **basic_params)},
    {"model": ExtraTreesRegressor(n_estimators=250, **basic_params)},
    {"model": ExtraTreesRegressor(n_estimators=200, **basic_params)},
]

importance = round1(models, train_size=0.95, score_func=f_regression, percentile=100)

                         Starting round 1                         
------------------------------------------------------------------


100%|██████████| 4/4 [00:37<00:00,  9.28s/it]

================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features='log2', rando...       1.0   0.965819
1  (ExtraTreeRegressor(max_features='log2', rando...       1.0   0.965409
2  (ExtraTreeRegressor(max_features='log2', rando...       1.0   0.965212
3  (ExtraTreeRegressor(max_features='log2', rando...       1.0   0.964918
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features='log2', n_estimators=250, n_jobs=-1)
------------------------------------------------------------------
================================ Feature Importance ================================
                      Features  Importance  cumulative_importance
0                    atran_std    0.016093                   0.02
1                     drad_min    0.015748           

In [ ]:
best_model, X_train = round2(
    models, pareto_threshold=0.4, importance=importance, train_size=0.999
)

================================ Pareto Features ================================
              Features  Importance  cumulative_importance
0            atran_std    0.016093                   0.02
1             drad_min    0.015748                   0.03
2      apparent_zenith    0.015272                   0.05
3   apparent_elevation    0.014962                   0.06
4            elevation    0.014231                   0.08
5            urad_mean    0.014163                   0.09
6     airmass_absolute    0.013925                   0.10
7             urad_std    0.013807                   0.12
8     airmass_relative    0.013068                   0.13
9               zenith    0.012918                   0.14
10            urad_max    0.011849                   0.16
11            drad_var    0.011805                   0.17
12                 ghi    0.011782                   0.18
13          atran_mean    0.011779                   0.19
14            drad_std    0.011704              

100%|██████████| 4/4 [00:20<00:00,  5.15s/it]

================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features='log2', rando...       1.0   0.959618
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features='log2', n_estimators=300, n_jobs=-1)
------------------------------------------------------------------
================================ Feature Importance ================================
              Features  Importance  cumulative_importance
0            atran_var    0.034855                   0.03
1               zenith    0.033468                   0.07
2            atran_std    0.032316                   0.10
3             urad_max    0.032289                   0.13
4     airmass_absolute    0.032122                   0.17
5             urad_var    0.031772                   0.20
6

# Predicting Submission

In [ ]:
import pickle
from google.colab import files

model = "best_model.pkl"
with open(model, "wb") as f:
    pickle.dump(best_model.model, f)
files.download(model)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Done downloading model!")

Done downloading model!


In [ ]:
with open(model, "rb") as f:
    pickled_model = pickle.load(f)


def create_submission(filename: str, model):
    sub_df = load_data("Submission")
    final_df = pd.read_csv(
        "/content/drive/MyDrive/EY 2025/Submission_Final.csv",
        usecols=["Latitude", "Longitude"],
    )

    print("Predicting", sub_df.shape[0], "rows...")
    to_predict = sub_df.loc[:, X_train.columns]
    to_predict.to_csv("validation_set.csv", index=False)
    final_df["UHI Index"] = model.predict(to_predict)
    final_df.to_csv(filename, index=False)
    print("Done!")
    return


sub_file = "submission.csv"
create_submission(sub_file, pickled_model)
files.download(sub_file)
sub_output = pd.read_csv(sub_file)
sub_output

Predicting 1040 rows...
Done!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Longitude,Latitude,UHI Index
0,-73.971665,40.788763,0.963732
1,-73.971928,40.788875,0.964016
2,-73.967080,40.789080,0.963634
3,-73.972550,40.789082,0.962666
4,-73.969697,40.787953,0.959309
...,...,...,...
1035,-73.919388,40.813803,1.041917
1036,-73.931033,40.833178,1.042770
1037,-73.934647,40.854542,1.040329
1038,-73.917223,40.815413,1.039372


In [ ]:
!rm *.csv

---